# Project 3: Retail Demand Forecasting - V4 (The Perfect Fit)

In V3, LightGBM beautifully captured the highly volatile intermittent demand. However, **Facebook Prophet** was still struggling with the Seasonal item. 

## V4 Upgrades: Understanding Prophet's True Purpose
Prophet was built for **macro-level, continuous data** (like total store sales), not micro-level sparse data (a single item that sells 0 units on most days). 

To make Prophet shine, we are changing the **Seasonal Demand** profile from a single intermittent item to the **Aggregated Total Sales of the Hobbies Department**. By aggregating the items, the structural zeros disappear, the curve becomes smooth, and Prophet can perfectly map the macro-seasonality.

In [ ]:
!pip install prophet lightgbm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

### 1. Load and Aggregate the Data
Notice how we aggregate the `HOBBIES_1` department for Prophet, but keep the individual items for SARIMAX and LightGBM.

In [ ]:
print("Loading M5 Dataset...")
df = pd.read_csv('/kaggle/input/m5-forecasting-accuracy/sales_train_validation.csv')

# 1. Steady Demand (Micro-level Item)
steady_item_id = 'FOODS_3_090_CA_1_validation'
steady_ts = df[df['id'] == steady_item_id].iloc[:, 6:].T

# 2. Seasonal Demand (MACRO-level Aggregation for Prophet)
hobbies_ca1 = df[(df['dept_id'] == 'HOBBIES_1') & (df['store_id'] == 'CA_1')]
seasonal_ts = hobbies_ca1.iloc[:, 6:].sum(axis=0).to_frame()

# 3. Volatile Demand (Micro-level Item)
volatile_item_id = 'HOUSEHOLD_1_118_CA_1_validation'
volatile_ts = df[df['id'] == volatile_item_id].iloc[:, 6:].T

dates = pd.date_range(start='2011-01-29', periods=1913, freq='D')
steady_ts.index = dates
seasonal_ts.index = dates
volatile_ts.index = dates

steady_ts.columns = ['Sales']
seasonal_ts.columns = ['Sales']
volatile_ts.columns = ['Sales']

# Visualize the difference between Micro and Macro data
fig, axes = plt.subplots(3, 1, figsize=(15, 10))
axes[0].plot(steady_ts[-365:], color='blue', label='Steady Demand (Micro Item)')
axes[0].legend()
axes[1].plot(seasonal_ts[-365:], color='green', label='Seasonal Demand (Macro Department Aggregation)')
axes[1].legend()
axes[2].plot(volatile_ts[-365:], color='red', label='Volatile Demand (Micro Item)')
axes[2].legend()
plt.tight_layout()
plt.show()

### 2. Advanced SARIMAX for Steady Demand

In [ ]:
train_steady = steady_ts[:-30]
test_steady = steady_ts[-30:]

print("Training Advanced SARIMAX...")
arima_model = ARIMA(train_steady, order=(5, 1, 2), seasonal_order=(1, 1, 1, 7))
arima_fit = arima_model.fit()

arima_preds = arima_fit.forecast(steps=30)

plt.figure(figsize=(10,4))
plt.plot(test_steady.index, test_steady['Sales'], label='Actual')
plt.plot(test_steady.index, arima_preds, color='red', label='SARIMAX Forecast')
plt.title('SARIMAX on Steady Demand (30-day forecast)')
plt.legend()
plt.show()

### 3. Prophet on MACRO Seasonal Data
Because the data is now a smooth aggregation of hundreds of items, Prophet's underlying Fourier series can finally map the trend perfectly without being destroyed by structural zeros.

In [ ]:
prophet_df = seasonal_ts.reset_index()
prophet_df.columns = ['ds', 'y']
train_seasonal = prophet_df[:-30]
test_seasonal = prophet_df[-30:]

print("Training Prophet on Aggregated Macro Data...")
m = Prophet(yearly_seasonality=True, weekly_seasonality=True)
m.fit(train_seasonal)

future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)

plt.figure(figsize=(10,4))
plt.plot(test_seasonal['ds'], test_seasonal['y'], label='Actual Aggregated Sales')
plt.plot(test_seasonal['ds'], forecast['yhat'][-30:], color='green', label='Prophet Forecast')
plt.title('Prophet on Macro Seasonal Demand (30-day forecast)')
plt.legend()
plt.show()

### 4. LightGBM (The Kaggle Winner) for Volatile Demand
We use LightGBM for the volatile micro-item, using feature engineering to give the trees explicit temporal memory.

In [ ]:
print("Engineering features for LightGBM...")
lgb_df = volatile_ts.copy()

# Date Features
lgb_df['day_of_week'] = lgb_df.index.dayofweek
lgb_df['is_weekend'] = lgb_df['day_of_week'].isin([5, 6]).astype(int)

# Lag Features
lgb_df['lag_1'] = lgb_df['Sales'].shift(1)
lgb_df['lag_7'] = lgb_df['Sales'].shift(7)
lgb_df['lag_28'] = lgb_df['Sales'].shift(28)

# Rolling Features
lgb_df['rolling_mean_7'] = lgb_df['Sales'].shift(1).rolling(7).mean()

lgb_df = lgb_df.dropna()

train_lgb = lgb_df[:-30]
test_lgb = lgb_df[-30:]

features = ['day_of_week', 'is_weekend', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7']
X_train = train_lgb[features]
y_train = train_lgb['Sales']
X_test = test_lgb[features]
y_test = test_lgb['Sales']

print("Training Kaggle-Winning LightGBM Model...")
model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

lgb_preds = model.predict(X_test)

plt.figure(figsize=(10,4))
plt.plot(test_lgb.index, y_test, label='Actual')
plt.plot(test_lgb.index, lgb_preds, color='purple', label='LightGBM Forecast')
plt.title('LightGBM (with Feature Engineering) on Volatile Demand (30-day forecast)')
plt.legend()
plt.show()